In [6]:
import json
from pathlib import Path

# ==========================================================
# Input / Output folders
# ==========================================================
INPUT_FOLDER = Path("./")
OUTPUT_FOLDER = Path("json")
OUTPUT_FOLDER.mkdir(exist_ok=True)

# ==========================================================
# Configuration for each dataset
# ==========================================================
CONFIG = {
    "kampongs": {
        "category": "Kampong",
        "name": "Name",
        "address": None,
        "postal": None,
        "Comments": None
    },

    "sg_religious_chinesetemples2010": {
        "category": "Chinese Temple",
        "name": "TEMPLE",
        "address": "ADD_",
        "postal": "POSTALCODE",
    },

    "sg_religious_chinesetemples_main": {
        "category": "Chinese Temple",
        "name": "Temple_Nam",
        "address": "Address",
        "postal": "POSTAL_COD",
    },

    "sg_religious_churches": {
        "category": "Church",
        "name": "CHURCH_NAM",
        "address": "CHURCH_ADD",
        "postal": "POSTAL_COD",
    },

    "sg_religious_clan_halls": {
        "category": "Clan Hall",
        "name": "NAME_OF_CL",
        "address": "ADDRESS",
        "postal": "POSTAL_COD",
    },

    "sg_religious_henghua": {
        "category": "Henghua Temple",
        "name": "BUILDING_N",
        "address": None,
        "postal": "POSTAL_COD",
    },

    "sg_religious_hindu_temples": {
        "category": "Hindu Temple",
        "name": "HINDU_TEMP",
        "address": "HINDU_ADD",
        "postal": "POSTAL_COD",
    },

    "sg_religious_lianhemiao_all": {
        "category": "United Temple",
        "name": "Official_N",
        "address": "Address",
        "postal": "Postal_Cod",
    },

    "sg_religious_lianhemiao_main": {
        "category": "United Temple",
        "name": "Official_N",
        "address": "Address",
        "postal": "Postal_Cod",
    },

    "sg_religious_mosques": {
        "category": "Mosque",
        "name": "Mosque_Nam",
        "address": "Mosque_Add",
        "postal": "POSTAL_COD",
    },
}


# ==========================================================
# Conversion
# ==========================================================

# ==========================================================
# Conversion
# ==========================================================

for file in INPUT_FOLDER.glob("*.geojson"):

    layer_name = file.stem

    if layer_name not in CONFIG:
        print(f"Skipping {layer_name} (no configuration)")
        continue

    cfg = CONFIG[layer_name]

    with open(file, "r", encoding="utf-8") as f:
        geojson = json.load(f)

    result = []

    for feature in geojson["features"]:

        props = feature.get("properties", {})
        geometry = feature.get("geometry", {})

        if geometry.get("type") != "Point":
            continue

        x, y = geometry["coordinates"]

        # Start with ALL original properties
        item = props.copy()

        # Add standardized fields
        item["id"] = props.get("OBJECTID")
        item["name"] = props.get(cfg["name"], "")
        item["category"] = cfg["category"]

        item["X"] = x
        item["Y"] = y

        item["address"] = (
            props.get(cfg["address"], "")
            if cfg["address"] else ""
        )

        item["postalCode"] = (
            props.get(cfg["postal"], "")
            if cfg["postal"] else ""
        )

        result.append(item)

    output_file = OUTPUT_FOLDER / f"{layer_name}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=4)

    print(f"{layer_name}: {len(result)} features converted.")

sg_religious_chinesetemples2010: 212 features converted.
kampongs: 227 features converted.
sg_religious_lianhemiao_main: 69 features converted.
sg_religious_henghua: 15 features converted.
sg_religious_churches: 234 features converted.
sg_religious_clan_halls: 109 features converted.
sg_religious_chinesetemples_main: 493 features converted.
sg_religious_mosques: 99 features converted.
sg_religious_lianhemiao_all: 297 features converted.
sg_religious_hindu_temples: 28 features converted.
